# Flat Valence Bands of Twisted Bilayer h-BN

> **Lede Xian, Dante M. Kennes, Nicolas Tancogne-Dejean, Massimo Altarelli, and Angel Rubio**
> **Multiflat Bands and Strong Correlations in Twisted Bilayer Boron Nitride: Doping-Induced Correlated Insulator and Superconductor**
> Nano Letters 19, 4934 (2019)
> [DOI: 10.1021/acs.nanolett.9b00986](https://doi.org/10.1021/acs.nanolett.9b00986)

Calculate the band structure of the twisted h-BN bilayer created in the
[structure notebook](interface_bilayer_twisted_commensurate_lattices_boron_nitride.ipynb), using Quantum ESPRESSO and the
band structure workflow from Standata, and read off the width of its top set of flat valence bands.

The article's result is that twisting bilayer h-BN splits its top valence bands into isolated flat sets whose width
shrinks with the twist angle, from 101.4 meV at 7.34° to 1.6 meV at 2.64° (Fig. 1g).

<h2 style="color:green">Usage</h2>

1. Create the material in the [structure notebook](interface_bilayer_twisted_commensurate_lattices_boron_nitride.ipynb), which saves it to the `uploads` folder under the name used in cell 1.2 below.
1. Set the material and the calculation parameters in cells 1.2 and 1.3 (or use the default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the results; section 8 prints the comparison with the paper.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material, workflow, model, compute resources and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the material by name from the `uploads` folder, print its provenance and save it to the platform.
1. Configure the workflow: select the application and the model, load the band structure workflow from Standata, and set the k-grid of the self-consistent run, the k-path of the bands and the cutoffs.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Run the job, re-using a job that already ran under the same workflow name.
1. Retrieve results: the band structure, the width of the top valence flat bands and their separation from the bands below, and the bands near the valence-band top.
1. Compare with Xian et al. (2019).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and jobs

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"
MATERIAL_NAME = "h-BN bilayer 7.34deg alpha d3.23"  # as saved by the structure notebook

WORKFLOW_SEARCH_TERM = "band_structure.json"
MY_WORKFLOW_NAME = "Band Structure"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OR
PPN = 16  # OR allows 16 cores per job
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.3. Set the DFT parameters

In [ ]:
MODEL_SUBTYPE = "lda"
FUNCTIONAL = "pz"
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV
ECUTWFC = 40  # Ry
ECUTRHO = 200  # Ry; paper: VASP PAW, 400 eV

KGRID = [1, 1, 1]  # Γ only, as the paper's self-consistent run
# paper: Γ–K–M, number of points not stated; ours 7, evenly spaced (|ΓK| = 2|KM|)
KPATH = [
    {"point": "Γ", "steps": 4},
    {"point": "K", "steps": 2},
    {"point": "M", "steps": 1},
]
MODEL_TAG = (f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KGRID[0]} "
             f"p{sum(point['steps'] for point in KPATH)}")

SCF_UNIT = "pw_scf"
BANDS_UNIT = "pw_bands"

PLOT_ENERGY_RANGE = (-0.5, 0.1)  # eV, relative to the top of the flat bands

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the material
### 3.1. Load from the uploads folder and print provenance

The structure, its geometry and its lattice type all come from the structure notebook; this one only loads it by
name.

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import load_material

material = load_material(client, FOLDER, MATERIAL_NAME, ACCOUNT_ID)
print(f"{material.name}: {material.basis.number_of_atoms} atoms, a = {material.lattice.a:.3f} Å, "
      f"gamma = {material.lattice.gamma:.3f}°, lattice {material.lattice.type}")

### 3.2. Preview the material

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

visualize_materials(material, viewer="wave")

### 3.3. Save the material to the platform

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

material.basis.set_labels_from_list([])
saved_material = Material.create(get_or_create_material(client, material, ACCOUNT_ID))

## 4. Configure the workflow
### 4.1. Select application and model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. Configure the workflow

The self-consistent run on `KGRID`, then the bands along `KPATH`, both with the cutoffs of 1.3. The workflow is named
after the material and `MODEL_TAG`; section 6 finds a job that already ran by that name.

The structure is not relaxed, as in the article's band-width series.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider, \
    PointsPathDataProvider

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = f"{MY_WORKFLOW_NAME} {MATERIAL_NAME} {MODEL_TAG}"
subworkflow = workflow.subworkflows[0]
subworkflow.model = model

cutoffs_context = PlanewaveCutoffsContextProvider(
    wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).get_context_item_data()
grid_context = PointsGridDataProvider(material=material, dimensions=KGRID, isEdited=True).get_context_item_data()
path_context = PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data()

for unit_name, contexts in [(SCF_UNIT, [grid_context, cutoffs_context]),
                            (BANDS_UNIT, [path_context, cutoffs_context])]:
    unit = subworkflow.get_unit_by_name(name=unit_name)
    for context in contexts:
        unit.add_context(context)
    subworkflow.set_unit(unit)
print(workflow.name)

### 4.3. Preview the workflow

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

visualize_workflow(workflow)

### 4.4. Save the workflow to the collection

In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflow = Workflow.create(get_or_create_workflow(client, workflow, ACCOUNT_ID))
print(f"{MATERIAL_NAME}: workflow {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the job

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Run the job

The job is found by its material and its workflow name, which carries the model tag, and re-used if it exists;
otherwise it is created and submitted.

In [ ]:
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

job = find_job_for_material(
    client, saved_material.id, workflow.name, ACCOUNT_ID, statuses=("submitted", "queued", "active", "finished"),
)
if job is None:
    job = create_job(
        api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
        owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
    )
    submit_jobs(client.jobs, [job["_id"]])
    print(f"✅ Submitted job {job['_id']}")
else:
    print(f"♻️  Reusing job {job['_id']}")
job_id = job["_id"]
await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 7. Retrieve the results
### 7.1. Band structure

The path is Γ–K–M of the moiré cell, as in the article; the bands of the two layers are folded onto its small
Brillouin zone.

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

band_structure = get_properties_for_job(client, job_id, property_name="band_structure")[0]
visualize_properties(band_structure, title=f"Band Structure: {MATERIAL_NAME}",
                     extra_config={"material": material.to_dict()})

### 7.2. Top valence flat bands

The flat set starts at the highest occupied band and takes in each band below whose maximum along the path reaches
the set's minimum. Its width is the spread of the set over the points of `KPATH`, and its separation is the distance
from the set's minimum down to the maximum of the next band. The article does not say whether its width is taken along
the path or over the whole Brillouin zone.

In [ ]:
import numpy as np

NUMBER_OF_OCCUPIED_BANDS = 2 * material.basis.number_of_atoms  # 8 valence electrons per B–N pair: B 3, N 5

bands = np.array(band_structure["yDataSeries"])  # band × path point, eV
flat_set_bottom = NUMBER_OF_OCCUPIED_BANDS - 1
while bands[flat_set_bottom - 1].max() >= bands[flat_set_bottom:NUMBER_OF_OCCUPIED_BANDS].min():
    flat_set_bottom -= 1
flat_set = bands[flat_set_bottom:NUMBER_OF_OCCUPIED_BANDS]
flat_set_top = flat_set.max()
flat_band_width = 1000 * (flat_set_top - flat_set.min())
flat_band_separation = 1000 * (flat_set.min() - bands[flat_set_bottom - 1].max())

### 7.3. Bands near the valence-band top (Fig. 1f of the paper)

Energies relative to the top of the flat set, which is drawn in red.

In [ ]:
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

vertex_indices = np.cumsum([0] + [point["steps"] for point in KPATH[:-1]])
figure, axes = plt.subplots()
axes.plot((bands - flat_set_top).T, color="black", marker=".")
axes.plot((flat_set - flat_set_top).T, color="red", marker=".")
axes.set_xticks(vertex_indices, [point["point"] for point in KPATH])
axes.set_ylim(*PLOT_ENERGY_RANGE)
axes.set_ylabel("E − E(top of the flat bands) (eV)")
axes.set_title(f"Bands near the valence-band top: {MATERIAL_NAME}")
display_matplotlib_figure(figure)

## 8. Compare with Xian et al. (2019)

The article's width at 7.34° beside ours; the article gives no separation at this angle.

Settings that differ from the article's: GBRV ultrasoft pseudopotentials at 40/200 Ry against VASP PAW at 400 eV, and
Standata's lattice constant of 2.509 Å against 2.512 Å.

In [ ]:
XIAN_2019_FLAT_BAND_WIDTH = 101.4  # meV, 7.34°, text p. 4936 and Fig. 1g

print(f"Top valence flat-band width along Γ–K–M: {flat_band_width:.1f} meV · "
      f"Xian et al. 2019: {XIAN_2019_FLAT_BAND_WIDTH} meV")
print(f"Separation from the bands below: {flat_band_separation:.1f} meV")

## References

[1] L. Xian, D. M. Kennes, N. Tancogne-Dejean, M. Altarelli and A. Rubio, "Multiflat Bands and Strong Correlations in Twisted Bilayer Boron Nitride: Doping-Induced Correlated Insulator and Superconductor", Nano Lett. 19, 4934 (2019). https://doi.org/10.1021/acs.nanolett.9b00986

[2] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053